# 智能旅行助手
在本案例中，我们的目标是构建一个能处理分步任务的智能旅行助手。需要解决的用户任务定义为："你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"要完成这个任务，智能体必须展现出清晰的逻辑规划能力。它需要先调用天气查询工具，并将获得的观察结果作为下一步的依据。在下一轮循环中，它再调用景点推荐工具，从而得出最终建议。

In [14]:
%pip install requests tavily-python openai

Note: you may need to restart the kernel to use updated packages.


### 工具 1：查询真实天气
我们将使用免费的天气查询服务 wttr.in，它能以 JSON 格式返回指定城市的天气数据。下面是实现该工具的代码：

In [36]:
import requests   # requests 是一个流行的 HTTP 客户端库，用于发送网络请求并处理响应

def get_weather(city: str) -> str:    # 返回值类型标注为 str，表示返回一个描述天气的字符串
    """
    通过调用第三方天气 API（wttr.in）查询真实的天气信息。
    API格式：https://wttr.in/{city}?format=j1   
    """
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"  #  格式化字符串  支持变量插值
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        weather_data = response.json()
        print("得到的天气数据:", weather_data)   
        print("============================"*2)    # 这两行将完整的 JSON 数据打印到控制台，仅用于调试。在生产环境中应删除或改用日志（logging.debug），避免暴露大量数据或污染标准输出。    
        # 获取当前天气信息
        # weather_desc = weather_data['current_condition'][0]['weatherDesc'][0]['value']
        # return f"{city}的天气是: {weather}"

        
        # 提取当前天气状况
        current_condition = weather_data['current_condition'][0]   # 从 JSON 中逐层提取数据 是一个列表（通常长度为 1），取第一个元素
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时发出请求，遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"


# 这段代码定义了一个名为 get_weather 的函数，它通过调用第三方天气 API（wttr.in）查询指定城市的实时天气状况，
# 并将结果以自然语言字符串形式返回。

In [ ]:
# 测试上面的函数

city = '长沙'
get_weather(city)

得到的天气数据: {'current_condition': [{'FeelsLikeC': '39', 'FeelsLikeF': '103', 'cloudcover': '56', 'humidity': '75', 'observation_time': '07:41 AM', 'precipInches': '0.1', 'precipMM': '2.2', 'pressure': '1001', 'pressureInches': '30', 'temp_C': '31', 'temp_F': '88', 'uvIndex': '5', 'visibility': '10', 'visibilityMiles': '6', 'weatherCode': '353', 'weatherDesc': [{'value': 'Light rain shower'}], 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0009_light_rain_showers.png'}], 'winddir16Point': 'N', 'winddirDegree': '357', 'windspeedKmph': '19', 'windspeedMiles': '12'}], 'nearest_area': [{'areaName': [{'value': 'Changshashih'}], 'country': [{'value': 'China'}], 'latitude': '28.200', 'longitude': '112.967', 'population': '0', 'region': [{'value': 'Hunan'}], 'weatherUrl': [{'value': 'https://www.worldweatheronline.com/v2/weather.aspx?q=28.2,112.967'}]}], 'request': [{'query': 'Lat 28.19 and Lon 112.98', 'type': 'LatLon'}], 'weather': [{'astronomy'

'长沙当前天气:Light rain shower，气温31摄氏度'

<img src = "./2.png" width="700"/>

### 工具 2：搜索并推荐旅游景点
我们将定义一个新工具 get_attraction，它会根据城市和天气状况，在互联网上搜索合适的景点：

In [42]:
import os        # os：标准库，用于读取环境变量 TAVILY_API_KEY
from tavily import TavilyClient   # tavily：第三方库，需要安装（pip install tavily-python）。它封装了 Tavily Search API 的调用，提供简洁的客户端接口。
                                # 导入Tavily Search API客户端类,它是一个供ai调用的搜索接口，请先在tavily.com注册一个账号，并获取API密钥，然后在环境变量中配置TAVILY_API_KEY
                               
def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端  使用密钥实例化 TavilyClient，后续通过该对象调用搜索方法
    tavily = TavilyClient(api_key=key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    # 长沙在Patchy rain nearby, 气温31摄氏度天气下最值得去的旅游景点推荐及理由
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答  这是 Tavily 的特色功能
        response = tavily.search(query, search_depth="basic", include_answer=True)
        # search_depth="basic"：表示使用基础搜索模式（相对 "advanced" 更轻量，消耗更少积分） 可以通过对search按ctrl查看具体参数说明


        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果 
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")  
        # 从 response['results'] 列表中提取每条结果的标题和内容摘要（content 可能是截取的相关段落），组装成无序列表
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"


# 这段代码定义了一个名为 get_attraction(city, weather) 的函数，它利用 Tavily Search API（一个为 AI 应用优化的搜索引擎）
# 根据给定的城市和天气状况，搜索并返回该城市在特定天气下值得一游的景点推荐。
# 函数最终返回一个自然语言的回答或一系列搜索结果摘要。

In [18]:
# 测试上面的函数
get_attraction('长沙', 'Patchy rain nearby, 气温31摄氏度')


'In Patchy rain and 31°C weather, visit Yuelu Mountain for a scenic hike and cultural experience. Try local delicacies like spicy shrimp and stinky tofu. Best time is April to May for pleasant weather.'

In [38]:
# 组装以上的工具函数， 供LLM使用
available_tools = {     # 采用字典（Dict）结构，键值对的方式
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

### 接入大语言模型
当前，许多 LLM 服务提供商（包括 OpenAI、Azure、以及众多开源模型服务框架如 Ollama、vLLM 等）都遵循了与 OpenAI API 相似的接口规范。这种标准化为开发者带来了极大的便利。智能体的自主决策能力来源于 LLM。我们将实现一个通用的客户端 OpenAICompatibleClient，它可以连接到任何兼容 OpenAI 接口规范的 LLM 服务。

In [39]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容OpenAI接口的LLM服务的客户端。
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.api_key = api_key
        self.base_url = base_url
        self.client = OpenAI(
            api_key=self.api_key, 
            base_url=self.base_url
        )

    def generate(self, prompt: str, system_prompt: str = None):
        """
            prompt: 用户输入提示词
            system_prompt: 系统提示词,用于引导模型的行为（角色描述，功能描述，约束条件等）
        """
        if not prompt:
            return "错误：请输入用户提示词"
        print("正在调用大语言模型...", self.model)
        if system_prompt:   # 若提供了 system_prompt，则消息为 [system, user]；否则只有 [user]
            messages = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': prompt}]
        else:
            messages = [{'role': 'user', 'content': prompt}]
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages = messages
            )
            print("原始响应:", response)
            print("============"*20)
            return response.choices[0].message.content  # 从 response.choices[0].message.content 提取生成的文本
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"

# 这段代码定义了一个名为 OpenAICompatibleClient 的 Python 类，
# 用于便捷地调用任何兼容 OpenAI API 格式的大语言模型（LLM）服务（如 OpenAI 官方、Azure OpenAI、本地部署的 vLLM、Ollama 等）。
# 类封装了 API 密钥、基础 URL 和模型名称，并提供 generate 方法，支持系统提示词（system prompt）和用户提示词（user prompt），
# 返回模型生成的文本。

In [24]:
# 测试以上函数
from openai import OpenAI
import os

open_ai_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("MODEL_NAME")
openai_base_url = os.getenv("OPENAI_BASE_URL")

client = OpenAICompatibleClient(api_key=open_ai_key, model=model_name,  base_url=openai_base_url)
result = client.generate("In Patchy rain and 31°C weather, visit Yuelu Mountain for a scenic hike and cultural experience. Try local delicacies like spicy shrimp and stinky tofu. Best time is April to May for pleasant weather")
print(result)


正在调用大语言模型... Qwen/Qwen3.5-35B-A3B
原始响应: ChatCompletion(id='chatcmpl-9541815c-bc8f-9a9c-9033-db749bcd0dc9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='# 🌧️ Yuelu Mountain: Misty Hike & Flavor Adventure Plan\n\nGiven your specified conditions (**31°C with patchy rain**) and your desire for a blend of culture, nature, and spice, here is a tailored itinerary to make the most of the weather while staying safe and comfortable.\n\nWhile **April and May** are indeed the "ideal" months for Yuelu Mountain, experiencing it under a light shower with high humidity can actually enhance the scenery, creating a **"mysterious ancient Taoist atmosphere"** often found in traditional Chinese paintings.\n\n---\n\n### ⚠️ Weather Reality Check\n*   **The Good:** The rain clears dust/mud, making the air crisp. The greenery is vibrant. Mist clinging to the peaks makes for stunning photography.\n*   **The Challenge:** 31°C with high humidity feels hotter

## 系统提示词
### 提示词工程-> context 工程-> harness工程

驱动真实 LLM 的关键在于提示词工程（Prompt Engineering）。我们需要设计一个“指令模板”，告诉 LLM 它应该扮演什么角色、拥有哪些工具、以及如何格式化它的思考和行动。这是我们智能体的“说明书”，它将作为system_prompt传递给 LLM。

In [43]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束
- 语言必须是中文.
- 必须严格按照条件进行回答。

请开始吧！
"""
# 提示词工程是非常重要的一步，它决定了智能体的行为和性能。

In [35]:
# 测试以上函数
from openai import OpenAI
import os

open_ai_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("MODEL_NAME")
openai_base_url = os.getenv("OPENAI_BASE_URL")

client = OpenAICompatibleClient(api_key=open_ai_key, model=model_name,  base_url=openai_base_url)
result = client.generate("In Patchy rain and 31°C weather, visit Yuelu Mountain for a scenic hike and cultural experience. Try local delicacies like spicy shrimp and stinky tofu. Best time is April to May for pleasant weather", AGENT_SYSTEM_PROMPT)
print(result)

正在调用大语言模型... Qwen/Qwen3.5-35B-A3B
原始响应: ChatCompletion(id='chatcmpl-5d046783-3f64-98c4-ab0f-48891e7cc1b4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Thought: 用户分享了关于长沙岳麓山的旅行建议，提到了天气条件和景点信息。我需要先确认城市的正确中文名称，然后查询实时天气和景点推荐来验证和补充用户的旅行计划。让我首先查询长沙的天气情况。\n\nAction: get_weather(city="长沙")', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, function_calls=None, reasoning_content='用户发送了一条关于旅行建议的信息，但这条信息看起来像是用户已经提供了旅行建议（包含天气、景点、美食和时间）。用户似乎是在分享一个旅行计划，或者希望我根据这个信息来提供进一步的帮助。\n\n让我分析一下用户的请求：\n- 提到天气条件：Patchy rain and 31°C\n- 推荐景点：Yuelu Mountain (岳麓山)\n- 推荐美食：spicy shrimp (辣虾) 和 stinky tofu (臭豆腐)\n- 最佳时间：April to May\n\n看起来用户可能想要我验证或补充这些信息，或者他们正在分享他们的旅行计划并希望得到回应。\n\n由于用户没有明确提出需要我使用工具做什么，我需要先确定城市名称来获取天气信息和景点推荐。从"Yuelu Mountain"可以判断这是长沙的著名景点。\n\n我应该先确认城市是长沙，然后可以查询天气并获取景点推荐。\n\n让我开始思考并执行工具调用。'), delta={'role': None, 'content': '', 'tool_calls': None, 'function_calls': None, 'reasoning_content': ''})]

### 执行行动循环，实现智能体的行为模式: Thought-Action-observe模式
下面的主循环将整合所有组件，并通过格式化后的Prompt驱动LLM进行决策。

In [41]:
# 测试以上函数
import os
import re
import requests
from openai import OpenAI
from tavily import TavilyClient


# --- 1. 配置LLM客户端 ---
# 请根据您使用的服务，将这里替换成对应的凭证和地址
open_ai_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("MODEL_NAME")
openai_base_url = os.getenv("OPENAI_BASE_URL")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")


llm = OpenAICompatibleClient(
    model=model_name,
    api_key=open_ai_key,
    base_url=openai_base_url
)

# --- 2. 初始化 ---
user_prompt = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]   # 初始化对话历史

print(f"用户输入: {user_prompt}\n" + "="*40)

# --- 3. 运行主循环 ---
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)

    # TODO: prompt如何压缩，如何用到缓存？
    
    # 3.2. 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break

    # 下一种情况：action是调用函数
    # Action: function_name(arg_name="arg_value")
    tool_name = re.search(r"(\w+)\(", action_str).group(1)    # 提取函数名
    args_str = re.search(r"\((.*)\)", action_str).group(1)    # 提取参数字符串: name="zy", age="20" => {name: zy, age: 20}
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
--- 循环 1 ---

正在调用大语言模型... Qwen/Qwen3.5-35B-A3B
原始响应: ChatCompletion(id='chatcmpl-5f14425a-2437-959b-bdb7-7c4d9366bcf8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Thought: 首先需要获取北京的实时天气信息，以便后续根据天气状况推荐景点。\nAction: get_weather(city="北京")', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, function_calls=None, reasoning_content='好的，用户让我查询北京的天气，然后根据天气推荐景点。首先我需要调用get_weather工具获取北京的天气信息。等拿到天气数据后，再根据天气情况使用get_attraction来推荐合适的景点。所以现在第一步应该是先查天气，之后根据结果再决定后续步骤。\n\n确认用户提到的城市是北京，天气是今天。所以正确调用get_weather(city="北京")。然后等待返回的天气情况，再进一步处理。如果天气是晴天，可能推荐户外景点；如果是雨天，可能推荐室内博物馆之类。但具体要看工具返回的信息结构，可能需要进一步分析。现在先执行第一步，获取天气数据。'), delta={'role': None, 'content': '', 'tool_calls': None, 'function_calls': None, 'reasoning_content': ''})], created=1787493135, model='Qwen/Qwen3.5-35B-A3B', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fi